In [1]:
%load_ext autoreload
%autoreload 2
import os

if os.getcwd().endswith("notebooks"):
    os.chdir("..")

print(os.getcwd()) # should end in /medjudge-audit

/Users/berniceyan/medjudge-audit


In [2]:
import pandas as pd
from judgeaudit.scoring import per_example_scores

b1 = pd.read_json("results/grades_track_b1.jsonl", lines=True)
ex1 = per_example_scores(b1)                     # one score per (model, conversation)

model_means = ex1.groupby("response_model").score.mean()
print(model_means)
gap = model_means.max() - model_means.min()
print(f"MODEL GAP: {gap:.3f}")

response_model
anthropic/claude-sonnet-4.5    0.478005
openai/gpt-4o-mini             0.370413
Name: score, dtype: float64
MODEL GAP: 0.108


In [4]:
import pandas as pd
b3 = pd.read_json("results/grades_track_b3.jsonl", lines=True)
piv = b3.pivot_table(index=["prompt_id", "response_model", "criterion_idx"], columns="run_tag", values="grade", aggfunc="first") 

flip_rate = (piv.nunique(axis=1) > 1).mean()
print(f"criteria whose verdict flips across reps: {flip_rate:.1%}")

criteria whose verdict flips across reps: 15.4%


In [5]:
# PILOT b2
b2 = pd.read_json("results/grades_track_b2.jsonl", lines=True)
print(b2.groupby(["judge_model", "variant"]).grade.apply(lambda s: s.isna().mean()))

judge_model              variant            
google/gemini-2.5-flash  v1_official            0.0
                         v2_terse               0.0
                         v3_clinical_persona    0.0
openai/gpt-4.1           v2_terse               0.0
                         v3_clinical_persona    0.0
Name: grade, dtype: float64


In [6]:
b1 = pd.read_json("results/grades_track_b1.jsonl", lines=True) 
one = b1[(b1.prompt_id == b1.prompt_id.iloc[0]) & 
         (b1.response_model == b1.response_model.iloc[0])] 
print(one[["criterion_idx", "points", "grade"]].to_string())

   criterion_idx  points  grade
0              0       5   True
1              1       5   True


In [7]:
from judgeaudit.scoring import example_score
print("function says:", example_score(one))

function says: 1.0
